In [1]:
import os, json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

API_KEY = os.getenv("NVIDIA_API_KEY")
if not API_KEY:
    raise RuntimeError(
        "NVIDIA_API_KEY not found. "
        "Copy .env.example to .env, paste your key into it, then restart the kernel."
    )

client = OpenAI(base_url="https://integrate.api.nvidia.com/v1", api_key=API_KEY)
MODEL = "nvidia/nemotron-3.5-lightning-30b-a3b"

# This is a *reasoning* model. Left on, it hands back its thinking process instead
# of the answer. We switch it off so `content` holds the actual reply.
NO_THINKING = {"chat_template_kwargs": {"enable_thinking": False}}

def ask(messages, temperature=0.7, **kwargs):
    """Send a message list to the model. Returns the assistant message object."""
    resp = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=temperature,
        max_tokens=2048,
        extra_body=NO_THINKING,
        **kwargs,
    )
    return resp.choices[0].message

print("Connected. Model:", MODEL)

Connected. Model: nvidia/nemotron-3.5-lightning-30b-a3b


In [2]:
import requests
from tavily import TavilyClient

WEATHER_CODES = {
    0: "clear sky", 1: "mainly clear", 2: "partly cloudy", 3: "overcast",
    45: "foggy", 48: "depositing rime fog",
    51: "light drizzle", 53: "moderate drizzle", 55: "dense drizzle",
    61: "slight rain", 63: "moderate rain", 65: "heavy rain",
    71: "slight snow", 73: "moderate snow", 75: "heavy snow",
    80: "slight rain showers", 81: "moderate rain showers", 82: "violent rain showers",
    95: "thunderstorm", 96: "thunderstorm with hail", 99: "thunderstorm with heavy hail",
}


def get_weather(city: str) -> dict:
    """Current weather for a city, via Open-Meteo (no API key needed)."""
    geo = requests.get("https://geocoding-api.open-meteo.com/v1/search",
                       params={"name": city, "count": 1}, timeout=15).json()
    if not geo.get("results"):
        return {"error": f"Could not find a place called '{city}'."}
    place = geo["results"][0]

    w = requests.get("https://api.open-meteo.com/v1/forecast", params={
        "latitude": place["latitude"], "longitude": place["longitude"],
        "current": "temperature_2m,relative_humidity_2m,precipitation,weather_code,wind_speed_10m",
        "timezone": "auto"}, timeout=15).json()["current"]

    return {
        "city": place["name"], "country": place.get("country", ""),
        "temperature_c": w["temperature_2m"], "humidity_percent": w["relative_humidity_2m"],
        "precipitation_mm": w["precipitation"], "wind_kmh": w["wind_speed_10m"],
        "conditions": WEATHER_CODES.get(w["weather_code"], "unknown"),
        "observed_at": w["time"],
    }


TAVILY_KEY = os.getenv("TAVILY_API_KEY")
tavily = TavilyClient(api_key=TAVILY_KEY) if TAVILY_KEY else None


def web_search(query: str, max_results: int = 3) -> dict:
    """Search the web via Tavily."""
    if tavily is None:
        return {"error": "TAVILY_API_KEY is not set. Get a free key at tavily.com and add it to .env."}
    response = tavily.search(query=query, max_results=max_results)
    return {"query": query, "results": [
        {"title": r["title"], "url": r["url"], "snippet": r["content"][:300]}
        for r in response["results"]]}


print("get_weather :", get_weather("Coimbatore")["temperature_c"], "°C")
print("web_search  :", "ready" if tavily else "no TAVILY_API_KEY (search cells will be skipped)")

get_weather : 29.4 °C
web_search  : ready


And the same two schemas:

In [3]:
weather_tool = {
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Get the current weather for a city. Use this whenever the user asks "
                       "about current temperature, rain, or conditions.",
        "parameters": {
            "type": "object",
            "properties": {"city": {"type": "string", "description": "The city name, e.g. 'Coimbatore'."}},
            "required": ["city"],
        },
    },
}

search_tool = {
    "type": "function",
    "function": {
        "name": "web_search",
        "description": "Search the web for current information, news, facts, or anything that "
                       "happened recently. Use when you do not know the answer.",
        "parameters": {
            "type": "object",
            "properties": {"query": {"type": "string", "description": "The search query."}},
            "required": ["query"],
        },
    },
}

TOOL_SCHEMAS = [weather_tool, search_tool]
print("2 tool schemas ready.")

2 tool schemas ready.


---

# 1. The realisation

Put Notebook 2 next to what we're about to write.

| **Notebook 2 — by hand** | **Notebook 3 — in a loop** |
|---|---|
| Step 3: call the model | `while True:` |
| Step 4–5: check `tool_calls` | ⤷ call the model |
| Step 6: *(you read a markdown cell)* | ⤷ `if not tool_calls: break` |
| Step 7: run the function | ⤷ run every requested function |
| Step 8: append the result | ⤷ append the results |
| Step 9: call the model again | ⤷ *(loop back to the top)* |

That's it. That is the entire conceptual leap.

In Notebook 2, if the model's second reply had *also* asked for a tool, you'd have been
stuck writing Steps 7, 8, 9 again. And again. The loop just... keeps going until the model
stops asking.

**Everything else — memory, planning, multi-step reasoning, "agentic behaviour" — falls out
of this loop for free.** Let's build it in five passes.

---

# 2. Building the loop, one piece at a time

## Pass 1 — The skeleton (no tools yet)

Start with a loop that doesn't loop. It calls the model once and breaks. Useless, but it's
the shape everything else hangs on.

In [ ]:
def agent_v1(user_message):
    messages = [{"role": "user", "content": user_message}]

    while True:
        response = client.chat.completions.create(
            model=MODEL, messages=messages, extra_body=NO_THINKING, max_tokens=1024,
        )
        message = response.choices[0].message

        # No tools wired up yet, so there is never anything to do. Always exits here.
        if not message.tool_calls:
            return message.content


print(agent_v1("What is the capital of Tamil Nadu?"))

The capital of Tamil Nadu is **Chennai**.


Works, but it's just `ask()` with extra steps. Now the interesting part.

## Pass 2 — The tool registry and the dispatch

The model returns a tool **name as a string**. We need to turn that string into an actual
Python function. A dictionary does that:

In [5]:
# The registry: maps the name in the schema -> the real Python function.
# This is the ONLY place the model's request touches your code.
TOOLS = {
    "get_weather": get_weather,
    "web_search": web_search,
}

print(TOOLS)

{'get_weather': <function get_weather at 0x00000256E1B154E0>, 'web_search': <function web_search at 0x00000256E18FBB00>}


This dictionary is your security boundary. The model can request `get_weather`. It cannot
request `os.system` or `delete_all_records`, because those aren't in here. **A model can
only reach what you put in this dict.**

Now the loop with dispatch:

In [6]:
# The model sometimes *announces* a tool instead of calling one ("Let me check the
# weather...") - which the loop below reads as a finished answer. One line of system
# prompt, and temperature=0 on the calls, keeps it honest.
AGENT_SYSTEM = (
    "You are a helpful assistant with access to tools. "
    "When a tool is needed, call it - never say that you are about to call one. "
    "Give your final answer only once you have the tool results you need."
)


def agent_v2(user_message):
    messages = [{"role": "system", "content": AGENT_SYSTEM},
                {"role": "user", "content": user_message}]

    while True:
        response = client.chat.completions.create(
            model=MODEL, messages=messages, tools=TOOL_SCHEMAS,
            temperature=0, extra_body=NO_THINKING, max_tokens=1024,
        )
        message = response.choices[0].message

        # The model gave a text answer -> we're done.
        if not message.tool_calls:
            return message.content

        # Otherwise: it wants tools. Record the request...
        messages.append(message.model_dump(exclude_none=True))

        # ...run each one, and append each result.
        for call in message.tool_calls:
            function = TOOLS[call.function.name]
            args = json.loads(call.function.arguments)

            result = function(**args)

            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": json.dumps(result),
            })
        # loop back to the top - the model now sees the results

print(agent_v2("What's the weather in Coimbatore?"))

The current weather in Coimbatore is **29.4 °C** with **63 % humidity**, a light **0.7 mm** of precipitation, and a wind speed of **15.6 km/h**. Conditions are **thunderstorm**.


**That is a working AI agent.** Twenty-odd lines.

Compare it to Notebook 2's ten separate cells — the code is *identical*, just arranged as a
loop instead of stepping through cells.

Note the `for call in message.tool_calls:` — the model can request several tools in one
turn, and this handles that. Notebook 2 only ever looked at `tool_calls[0]`.

The `AGENT_SYSTEM` line above and `temperature=0` are not decoration. Without them the
model will sometimes reply *"Let me check the weather..."* as **text** instead of emitting a
tool call - and the loop, seeing no `tool_calls`, treats that announcement as the final
answer. Telling the model to call rather than narrate is the cheapest fix there is.

## Pass 3 — A stop condition

`while True` with a network call inside should make you nervous. If the model gets stuck in
a pattern — searching, being unsatisfied, searching again — it will loop forever, and every
iteration costs money.

**Every agent needs a hard limit.**

In [7]:
def agent_v3(user_message, max_turns=6):
    messages = [{"role": "system", "content": AGENT_SYSTEM},
                {"role": "user", "content": user_message}]

    for turn in range(max_turns):          # <- bounded, not `while True`
        response = client.chat.completions.create(
            model=MODEL, messages=messages, tools=TOOL_SCHEMAS,
            temperature=0, extra_body=NO_THINKING, max_tokens=1024,
        )
        message = response.choices[0].message

        if not message.tool_calls:
            return message.content

        messages.append(message.model_dump(exclude_none=True))
        for call in message.tool_calls:
            result = TOOLS[call.function.name](**json.loads(call.function.arguments))
            messages.append({"role": "tool", "tool_call_id": call.id,
                             "content": json.dumps(result)})

    return f"⚠️ Stopped after {max_turns} turns without reaching an answer."


print(agent_v3("What's the weather in Madurai?"))

The current weather in Madurai is **clear sky** with a temperature of **39.1 °C**, humidity at **26 %**, and light wind of **9 km/h**. No precipitation is reported.


`max_turns` is not a nicety. It is the difference between a bug and a bill.

## Pass 4 — Make the reasoning visible

Right now the agent is a black box: question in, answer out. For *learning* — and for
debugging in production — you want to see each step.

In [8]:
def agent_v4(user_message, max_turns=6, verbose=True):
    messages = [{"role": "system", "content": AGENT_SYSTEM},
                {"role": "user", "content": user_message}]

    if verbose:
        print(f"👤 USER: {user_message}\n")

    for turn in range(1, max_turns + 1):
        response = client.chat.completions.create(
            model=MODEL, messages=messages, tools=TOOL_SCHEMAS,
            temperature=0, extra_body=NO_THINKING, max_tokens=1024,
        )
        message = response.choices[0].message

        if not message.tool_calls:
            if verbose:
                print(f"✅ TURN {turn} - final answer:\n")
            return message.content

        messages.append(message.model_dump(exclude_none=True))

        for call in message.tool_calls:
            args = json.loads(call.function.arguments)
            if verbose:
                print(f"🔧 TURN {turn} - calling {call.function.name}({args})")

            result = TOOLS[call.function.name](**args)

            if verbose:
                print(f"   ↩️  {json.dumps(result)[:150]}\n")

            messages.append({"role": "tool", "tool_call_id": call.id,
                             "content": json.dumps(result)})

    return f"⚠️ Stopped after {max_turns} turns."


print(agent_v4("Is it hotter in Chennai or Coimbatore right now?"))

👤 USER: Is it hotter in Chennai or Coimbatore right now?

🔧 TURN 1 - calling get_weather({'city': 'Chennai'})
   ↩️  {"city": "Chennai", "country": "India", "temperature_c": 34.5, "humidity_percent": 47, "precipitation_mm": 0.0, "wind_kmh": 4.5, "conditions": "partly

🔧 TURN 2 - calling get_weather({'city': 'Coimbatore'})
   ↩️  {"city": "Coimbatore", "country": "India", "temperature_c": 29.4, "humidity_percent": 63, "precipitation_mm": 0.7, "wind_kmh": 15.6, "conditions": "th

✅ TURN 3 - final answer:

Right now, Chennai is hotter: about **34.5 °C** compared to Coimbatore’s **29.4 °C**.


**Look at that trace.** The agent called `get_weather` twice — once per city — then compared
them. Nobody told it to do that. It worked out that answering the question required two
lookups.

That trace is the teaching device. The loop is doing the planning, one turn at a time.

## Pass 5 — Errors must not kill the loop

Real tools fail. Networks time out. APIs return 500. Users ask about cities that don't exist.

If a tool raises inside our loop, the whole agent crashes. But there's a much better option:
**hand the error back to the model as a tool result** and let it recover.

In [9]:
def run_agent(user_message, max_turns=6, verbose=True, tools=None, schemas=None):
    """The finished agent loop. This is the version we use for the rest of the notebook."""
    tools = tools or TOOLS
    schemas = schemas or TOOL_SCHEMAS
    messages = [{"role": "system", "content": AGENT_SYSTEM},
                {"role": "user", "content": user_message}]

    if verbose:
        print(f"👤 USER: {user_message}\n")

    for turn in range(1, max_turns + 1):
        response = client.chat.completions.create(
            model=MODEL, messages=messages, tools=schemas,
            temperature=0, extra_body=NO_THINKING, max_tokens=1024,
        )
        message = response.choices[0].message

        if not message.tool_calls:
            if verbose:
                print(f"✅ TURN {turn} - final answer:\n")
            return message.content

        messages.append(message.model_dump(exclude_none=True))

        for call in message.tool_calls:
            name = call.function.name

            try:
                args = json.loads(call.function.arguments)
                if name not in tools:
                    raise KeyError(f"No such tool: {name}")
                result = tools[name](**args)
            except Exception as e:
                # Do NOT crash. Report the failure to the model so it can adapt.
                result = {"error": f"{type(e).__name__}: {e}"}
                args = call.function.arguments

            if verbose:
                print(f"🔧 TURN {turn} - {name}({args})")
                print(f"   ↩️  {json.dumps(result)[:150]}\n")

            messages.append({"role": "tool", "tool_call_id": call.id,
                             "content": json.dumps(result)})

    return f"⚠️ Stopped after {max_turns} turns without a final answer."


print(run_agent("What's the weather in Xyzzyville?"))

👤 USER: What's the weather in Xyzzyville?

✅ TURN 1 - final answer:

I’m not sure where Xyzzyville is, so I can’t look up the weather. If you can tell me the city or region, I’ll be happy to check the current conditions for you.


The lookup failed, the error went back as a tool result, and the agent **explained the
problem instead of crashing**. It may even have retried with a corrected spelling.

That try/except is what separates a demo from something you'd deploy.

---

# 3. What the loop can do that Notebook 2 could not

Notebook 2's hand-written code did exactly **one** tool call. These need more.

## 3.1 Reasoning on top of a tool result

In [10]:
answer = run_agent("Should I carry an umbrella in Coimbatore today?")
print(answer)

👤 USER: Should I carry an umbrella in Coimbatore today?

🔧 TURN 1 - get_weather({'city': 'Coimbatore'})
   ↩️  {"city": "Coimbatore", "country": "India", "temperature_c": 29.4, "humidity_percent": 63, "precipitation_mm": 0.7, "wind_kmh": 15.6, "conditions": "th

✅ TURN 2 - final answer:

Based on the current weather data for Coimbatore:

- **Temperature:** 29.4°C (85°F)
- **Conditions:** Thunderstorm
- **Precipitation:** 0.7 mm (light rain)
- **Humidity:** 63%
- **Wind:** 15.6 km/h

Since there is already light precipitation (0.7 mm) and thunderstorm conditions, **yes, you should carry an umbrella** if you’ll be outdoors. The rain is light but present, and thunderstorms can bring sudden showers. Having an umbrella (or a raincoat) will keep you dry and provide some protection from any unexpected heavier bursts.


There is no `should_i_carry_an_umbrella` tool. The agent fetched raw weather data and
**reasoned** its way to a recommendation — combining a tool result with its own judgement.

## 3.2 Chaining — where the second call depends on the first

This is the example that proves the loop earns its keep.

In [11]:
answer = run_agent(
    "Find out which Indian city hosted the 2023 G20 summit, "
    "then tell me the current weather there."
)
print(answer)

👤 USER: Find out which Indian city hosted the 2023 G20 summit, then tell me the current weather there.

🔧 TURN 1 - web_search({'query': '2023 G20 summit Indian city host'})
   ↩️  {"query": "2023 G20 summit Indian city host", "results": [{"title": "2023 G20 New Delhi summit", "url": "https://en.wikipedia.org/wiki/2023_G20_New_De

🔧 TURN 2 - get_weather({'city': 'New Delhi'})
   ↩️  {"city": "New Delhi", "country": "India", "temperature_c": 35.0, "humidity_percent": 48, "precipitation_mm": 0.0, "wind_kmh": 4.0, "conditions": "main

✅ TURN 3 - final answer:

The 2023 G20 summit was hosted in **New Delhi**, India.  

Current weather in New Delhi (as of the latest observation):

- **Temperature:** 35 °C  
- **Conditions:** Mainly clear  
- **Humidity:** 48 %  
- **Precipitation:** 0 mm  
- **Wind:** 4 km/h  

Stay cool!


Read the trace carefully:

```
🔧 TURN 1 - web_search({'query': '2023 G20 summit host city India'})
   ↩️  ... New Delhi ...
🔧 TURN 2 - get_weather({'city': 'New Delhi'})
   ↩️  {"temperature_c": ...}
✅ TURN 3 - final answer
```

**The argument to Turn 2 did not exist when Turn 1 started.**

`"New Delhi"` came out of the search result. The agent had to run the first tool, read the
answer, and *then* work out what to ask the second tool. No single round trip can do this.
Notebook 2's manual code physically could not have produced this answer.

This is the moment "agent" stops being a buzzword: **the loop lets later steps depend on
earlier results.**

> Requires a Tavily key. Without one you'll see the search error — and notice the agent
> handles it gracefully rather than crashing.

## 3.3 Multiple tools, multiple turns

In [12]:
answer = run_agent(
    "Compare the weather in Coimbatore and Ooty, and tell me which is better "
    "for a day trip tomorrow. Search for any travel advisories too.",
    max_turns=8,
)
print(answer)

👤 USER: Compare the weather in Coimbatore and Ooty, and tell me which is better for a day trip tomorrow. Search for any travel advisories too.

🔧 TURN 1 - get_weather({'city': 'Coimbatore'})
   ↩️  {"city": "Coimbatore", "country": "India", "temperature_c": 29.4, "humidity_percent": 63, "precipitation_mm": 0.7, "wind_kmh": 15.6, "conditions": "th

🔧 TURN 2 - get_weather({'city': 'Ooty'})
   ↩️  {"city": "Udhagamandalam", "country": "India", "temperature_c": 18.5, "humidity_percent": 75, "precipitation_mm": 0.2, "wind_kmh": 10.1, "conditions":

🔧 TURN 3 - web_search({'query': 'travel advisories Coimbatore Ooty day trip September 2026'})
   ↩️  {"query": "travel advisories Coimbatore Ooty day trip September 2026", "results": [{"title": "Incredible India Travel & Tour Package", "url": "https:/

✅ TURN 4 - final answer:

**Current Weather (as of 13:30 today)**  

| City | Temperature | Humidity | Precipitation | Wind | Conditions |
|------|-------------|----------|---------------|------|--

---

# Series recap

**Notebook 1 —** A prompt is the complete text you send. Five components: instruction,
context, input data, output format, examples. Roles: `system`, `user`, `assistant`.
The model is stateless.

**Notebook 2 —** The model can't run code, so it *requests* calls and **you** dispatch them,
returning results with the `tool` role. And `response_format` turns "please use this schema"
into a guarantee.

**Notebook 3 —** Put that request/dispatch/return exchange in a bounded loop, and you have
an agent. The loop lets step *n+1* depend on step *n* — which is the only thing a single
round trip cannot do.

Nothing in these three notebooks was magic. It was a text predictor, a JSON schema, and a
`for` loop.